In [1]:
from workspace.sources.local_datasets.preprocessing.tokenization import NLTKTokenizer
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction, Lemmatization, Stemming
from workspace.sources.experiments.metrics import Loss
from workspace.sources.utils import class_name_to_str
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.local_datasets.preprocessing.utils import truncate
from sklearn.model_selection import ParameterGrid
from IPython.display import display, Markdown

from workspace.sources.local_datasets.preprocessing.encoders.encoders import GPT2_Encoder as Encoder
from workspace.sources.models.transformers.openai_gpt_models import GPT2 as Model


import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [2]:
experiment_name = f'prefinal-{class_name_to_str(Model.__name__)}-v2'
print('Experiment:', experiment_name)

Experiment: prefinal-gpt2-v2


In [4]:
def conduct_gpt_model_experiments(dataset_name,
                                  dataset_path,
                                  dataset_hparams_grid,
                                  model_hparams_grid):
    total_runs = len(dataset_hparams_grid) * len(model_hparams_grid)
    for i, dataset_params in enumerate(dataset_hparams_grid):
        encoder = dataset_params['preprocessings_pipeline'][-1]
        for j, model_hparams in enumerate(model_hparams_grid):
            run_number = i * len(model_hparams_grid) + j + 1
            display(Markdown(f'### Run {run_number} of {total_runs}'))
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model = Model(gpt_encoder=encoder,
                           train_best_model_metric=Loss,
                           training_arguments=model_hparams)

            recovery_experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model)
            recovery_experiment.run()


In [5]:

# max_tokens_params = [128, 512]
max_tokens_params = [128]

pipelines = []

for max_tokens in max_tokens_params:
    pipelines.extend([PreprocessingPipeline(name='minimal',
                                            iterable=[Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction',
                                            iterable=[NoiseReduction(),
                                                      Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Lemmatization(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_stemming',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Stemming(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)])
                      ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(pipelines, fraction=0.05)

dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': pipelines})

print('Dataset Hyperparameters Grid Size: ', len(dataset_hparams_grid))

# model_hparams_grid = ParameterGrid({'epochs': [10],
#                                     'batch_size': [16],
#                                     'learning_rate': [1e-05, 5e-05],
#                                     'lr_scheduler': ['linear'],
#                                     'optimizer': ['adamw_torch'],
#                                     'weight_decay': [0.0, 1e-3],
#                                     'early_stopping_patience': [3],
#                                     'early_stopping_threshold': [0.01]})

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16],
                                    'learning_rate': [5e-05],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})

print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


### ReCOVery Dataset

In [6]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid)

### Run 1 of 4

2025-05-16 02:05:34,818 - Experiment - INFO - MLflow experiment initialized with ID: 649077292658744800
2025-05-16 02:05:34,819 - Experiment - INFO - Model signature: model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none)
2025-05-16 02:05:34,820 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),gpt2__encoder(t=gpt2,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 02:05:34,820 - Experiment - INFO - Run signature: model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),gpt2__encoder(t=gpt2,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 02:05:34,944 - Experiment - INFO - Found existing run with signature model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,es

### Run 2 of 4

2025-05-16 02:05:34,953 - Experiment - INFO - MLflow experiment initialized with ID: 649077292658744800
2025-05-16 02:05:34,954 - Experiment - INFO - Model signature: model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none)
2025-05-16 02:05:34,954 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),noise_reduction(),gpt2__encoder(t=gpt2,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 02:05:34,955 - Experiment - INFO - Run signature: model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),noise_reduction(),gpt2__encoder(t=gpt2,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 02:05:35,072 - Experiment - INFO - Found existing run with signature model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05

### Run 3 of 4

...

2025-05-16 02:06:44,594 - Experiment - INFO - Test metrics: {'test_loss': 0.6964749693870544, 'test_accuracy': 0.6, 'test_precision': 0.6, 'test_recall': 1.0, 'test_f1_score': 0.75, 'test_roc_auc': 0.5185185185185185, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1819, 'test_samples_per_second': 82.448, 'test_steps_per_second': 5.497, 'test_epoch': 1.0}
2025-05-16 02:06:44,613 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:06:44,615 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:06:44,653 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:06:44,654 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 02:06:44,655 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-5
2025-05-16 02:06:45,169 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7139406800270081, 'eval_accuracy': 0.6, 'eval_precision': 0.64285714

2025-05-16 02:06:45,297 - Experiment - INFO - Test metrics: {'test_loss': 0.6964749693870544, 'test_accuracy': 0.6, 'test_precision': 0.6, 'test_recall': 1.0, 'test_f1_score': 0.75, 'test_roc_auc': 0.5185185185185185, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1229, 'test_samples_per_second': 122.004, 'test_steps_per_second': 8.134, 'test_epoch': 1.0}
2025-05-16 02:06:45,569 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:06:45,572 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:06:45,650 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:06:45,653 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 02:06:45,654 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-10
2025-05-16 02:06:46,511 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7120739817619324, 'eval_accuracy': 0.4666666666666667,

2025-05-16 02:06:46,639 - Experiment - INFO - Test metrics: {'test_loss': 0.6792124509811401, 'test_accuracy': 0.4, 'test_precision': 0.5, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.5714285714285714, 'test_roc_auc': 0.5740740740740741, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.122, 'test_samples_per_second': 122.985, 'test_steps_per_second': 8.199, 'test_epoch': 2.0}
2025-05-16 02:06:46,657 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:06:46,659 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:06:46,702 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:06:46,703 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 02:06:46,704 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-15
2025-05-16 02:06:47,329 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7155303359031677, 'eval_

2025-05-16 02:06:47,463 - Experiment - INFO - Test metrics: {'test_loss': 0.683748185634613, 'test_accuracy': 0.4, 'test_precision': 0.5, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.5714285714285714, 'test_roc_auc': 0.537037037037037, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.1285, 'test_samples_per_second': 116.75, 'test_steps_per_second': 7.783, 'test_epoch': 3.0}
2025-05-16 02:06:47,478 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:06:47,479 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:06:47,518 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:06:47,520 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 02:06:47,521 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-10
2025-05-16 02:06:48,013 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7120739817619324, 'eval_accur

2025-05-16 02:06:48,144 - Experiment - INFO - Test metrics: {'test_loss': 0.6792124509811401, 'test_accuracy': 0.4, 'test_precision': 0.5, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.5714285714285714, 'test_roc_auc': 0.5740740740740741, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.1252, 'test_samples_per_second': 119.822, 'test_steps_per_second': 7.988, 'test_epoch': 2.0}
2025-05-16 02:06:48,165 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:06:48,168 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:06:48,210 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:06:48,212 - Experiment - INFO - Finished model evaluations stage.


### EUvsDisinfo Dataset

In [ ]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid)

### ISOT Dataset

In [ ]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid)

### CZ Dataset

In [7]:
cz_pipelines = []

for max_tokens in max_tokens_params:
    cz_pipelines.extend([PreprocessingPipeline(name='minimal',
                                               iterable=[Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction',
                                               iterable=[NoiseReduction(),
                                                         Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Lemmatization(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_stemming',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Stemming(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)])
                         ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(cz_pipelines, fraction=0.15)

cz_dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': cz_pipelines})
print('Dataset Hyperparameters Grid Size: ', len(cz_dataset_hparams_grid))
print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(cz_dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


In [8]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_gpt_model_experiments(dataset_name,
                              dataset_path,
                              cz_dataset_hparams_grid,
                              model_hparams_grid)

### Run 1 of 4

2025-05-16 02:06:56,742 - Experiment - INFO - MLflow experiment initialized with ID: 649077292658744800
2025-05-16 02:06:56,743 - Experiment - INFO - Model signature: model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none)
2025-05-16 02:06:56,744 - Experiment - INFO - Dataset signature: dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.15,tt=top),gpt2__encoder(t=gpt2,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 02:06:56,744 - Experiment - INFO - Run signature: model(mn=gpt2,ti=gpt2,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01,pt=none,pti=none);dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.15,tt=top),gpt2__encoder(t=gpt2,t=true,p=max_length,tml=128,isiw=false,pt=none)])
2025-05-16 02:06:57,743 - Experiment - INFO - Run ID: 1816c3f5938941daaaf8a03c96e16c56
2025-05-16 02:06:57,744 - Experiment - INFO - Run name: model(mn=gpt2,ti=gpt2,mmn=

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 02:10:17,878 - Experiment - INFO - Test metrics: {'test_loss': 0.7663020491600037, 'test_accuracy': 0.5, 'test_precision': 0.625, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.6451612903225806, 'test_roc_auc': 0.48571428571428577, 'test_false_positives_rate': 0.8571428571428571, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.2532, 'test_samples_per_second': 86.881, 'test_steps_per_second': 7.898, 'test_epoch': 6.0}
2025-05-16 02:10:17,898 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:10:17,901 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:10:17,958 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:10:17,959 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 02:10:17,960 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-21
2025-05-16 02:10:18,456 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.73713

2025-05-16 02:10:18,638 - Experiment - INFO - Test metrics: {'test_loss': 0.7602422833442688, 'test_accuracy': 0.6363636363636364, 'test_precision': 0.6842105263157895, 'test_recall': 0.8666666666666667, 'test_f1_score': 0.7647058823529411, 'test_roc_auc': 0.4761904761904762, 'test_false_positives_rate': 0.8571428571428571, 'test_false_negatives_rate': 0.13333333333333333, 'test_runtime': 0.1757, 'test_samples_per_second': 125.205, 'test_steps_per_second': 11.382, 'test_epoch': 3.0}
2025-05-16 02:10:18,654 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:10:18,656 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:10:18,715 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:10:18,716 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 02:10:18,717 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-42
2025-05-16 02:10:19,217 - Experiment - INFO - Best entry according to

2025-05-16 02:10:19,409 - Experiment - INFO - Test metrics: {'test_loss': 0.7663020491600037, 'test_accuracy': 0.5, 'test_precision': 0.625, 'test_recall': 0.6666666666666666, 'test_f1_score': 0.6451612903225806, 'test_roc_auc': 0.48571428571428577, 'test_false_positives_rate': 0.8571428571428571, 'test_false_negatives_rate': 0.3333333333333333, 'test_runtime': 0.1863, 'test_samples_per_second': 118.065, 'test_steps_per_second': 10.733, 'test_epoch': 6.0}
2025-05-16 02:10:19,424 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:10:19,426 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:10:19,483 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:10:19,484 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 02:10:19,484 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-21
2025-05-16 02:10:20,005 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7371

2025-05-16 02:10:20,195 - Experiment - INFO - Test metrics: {'test_loss': 0.7602422833442688, 'test_accuracy': 0.6363636363636364, 'test_precision': 0.6842105263157895, 'test_recall': 0.8666666666666667, 'test_f1_score': 0.7647058823529411, 'test_roc_auc': 0.4761904761904762, 'test_false_positives_rate': 0.8571428571428571, 'test_false_negatives_rate': 0.13333333333333333, 'test_runtime': 0.1851, 'test_samples_per_second': 118.841, 'test_steps_per_second': 10.804, 'test_epoch': 3.0}
2025-05-16 02:10:20,214 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:10:20,216 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:10:20,271 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:10:20,272 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 02:10:20,273 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-21
2025-05-16 02:10:20,802 - Experiment - INFO - Best entry according to validation metr

2025-05-16 02:10:21,003 - Experiment - INFO - Test metrics: {'test_loss': 0.7602422833442688, 'test_accuracy': 0.6363636363636364, 'test_precision': 0.6842105263157895, 'test_recall': 0.8666666666666667, 'test_f1_score': 0.7647058823529411, 'test_roc_auc': 0.4761904761904762, 'test_false_positives_rate': 0.8571428571428571, 'test_false_negatives_rate': 0.13333333333333333, 'test_runtime': 0.1909, 'test_samples_per_second': 115.233, 'test_steps_per_second': 10.476, 'test_epoch': 3.0}
2025-05-16 02:10:21,030 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:10:21,033 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:10:21,109 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:10:21,114 - Experiment - INFO - Finished model evaluations stage.
